# Notebook 07 — Gap Penalties: Linear vs. Affine

**Module:** 08 — Bioinformatics: Sequence Analysis  
**Notebook:** 07 of 18  
**Time estimate:** 75 minutes

> Affine gap penalty is the production standard. The 3-matrix DP is the
> key implementation concept — master it here before BLAST (NB08).

---
## Step 1 — Motivation

Linear gap penalty ($d$ per position) penalises a 10-gap insertion as 10× a 1-gap
insertion. But biologically, a single insertion event of length 10 is not 10× less
likely than a length-1 insertion — once the gap has opened, extending it is cheap.

Affine gap penalty separates: **gap open** (costly — a new indel event) and **gap
extend** (cheap — extending an existing indel). This produces biologically more
realistic alignments. BLAST, CLUSTAL, and virtually all production aligners use
affine gap penalties.

---
## Step 2 — Intuition

Linear: cost of a gap of length $k$ = $k \cdot d$.  
Affine: cost of a gap of length $k$ = $g_o + (k-1) \cdot g_e$, where $g_o$ is the
open penalty (always negative, large magnitude) and $g_e$ is the extend penalty
(negative, small magnitude).

Typical values (BLAST protein): $g_o = -11$, $g_e = -1$.  
A gap of length 3: $-11 + 2 \times (-1) = -13$.  
Compared to three length-1 gaps: $3 \times -11 = -33$ with affine (correct: one event),
or $3 \times -2 = -6$ with linear (incorrect: no disincentive to fragment).

The computational problem: to implement affine gap penalty correctly, you need **three
DP matrices** running in parallel.

---
## Step 3 — Biological Background

Insertions and deletions in proteins and DNA occur as molecular events:
- Slippage during replication
- Transposable element insertion
- Recombination-mediated indels

Each such event creates one gap. The length of the gap is determined by the event, not
by repeated single-nucleotide steps. Therefore, a long gap is one event — and the
opening cost should be paid once, not per position.

Affine gap penalties also tend to produce alignments where gaps cluster together rather
than fragmenting — which better reflects insertion-deletion biology.

---
## Step 4 — Mathematical Explanation

**Three matrices:**
- $M[i][j]$: best score of alignment ending with a match/mismatch at positions $(i,j)$
- $I_x[i][j]$: best score of alignment ending with a gap in $seq_1$ (i.e., char from $seq_2$ aligned to gap)
- $I_y[i][j]$: best score of alignment ending with a gap in $seq_2$ (i.e., char from $seq_1$ aligned to gap)

**Recurrences:**

$$M[i][j] = s(i,j) + \max(M[i-1][j-1],\ I_x[i-1][j-1],\ I_y[i-1][j-1])$$

$$I_x[i][j] = \max(M[i-1][j] + g_o,\ I_x[i-1][j] + g_e)$$

$$I_y[i][j] = \max(M[i][j-1] + g_o,\ I_y[i][j-1] + g_e)$$

**Final score:** $\max(M[n][m],\ I_x[n][m],\ I_y[n][m])$

**Key:** $I_x[i-1][j] + g_e$ means "we were already in a gap in seq1 at the previous
row, and we extend it" — only $g_e$ is paid, not $g_o$ again.

In [ ]:
# Step 6 — Python Implementation
import numpy as np
from typing import Tuple

NEG_INF = -10**9


def affine_gap_nw(
    seq1: str,
    seq2: str,
    match: int = 1,
    mismatch: int = -1,
    gap_open: int = -2,   # penalty for starting a gap (negative)
    gap_extend: int = -1  # penalty for extending a gap (negative, smaller magnitude)
) -> Tuple[int, str, str]:
    n, m = len(seq1), len(seq2)

    # Three matrices
    M  = np.full((n + 1, m + 1), NEG_INF, dtype=float)
    Ix = np.full((n + 1, m + 1), NEG_INF, dtype=float)  # gap in seq1
    Iy = np.full((n + 1, m + 1), NEG_INF, dtype=float)  # gap in seq2

    # Initialize
    M[0, 0] = 0
    for i in range(1, n + 1):
        Ix[i, 0] = gap_open + (i - 1) * gap_extend
        Iy[i, 0] = NEG_INF
    for j in range(1, m + 1):
        Iy[0, j] = gap_open + (j - 1) * gap_extend
        Ix[0, j] = NEG_INF

    # Fill
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sc = match if seq1[i - 1] == seq2[j - 1] else mismatch

            M[i, j] = sc + max(
                M[i - 1, j - 1],
                Ix[i - 1, j - 1],
                Iy[i - 1, j - 1]
            )

            Ix[i, j] = max(
                M[i - 1, j] + gap_open,
                Ix[i - 1, j] + gap_extend
            )

            Iy[i, j] = max(
                M[i, j - 1] + gap_open,
                Iy[i, j - 1] + gap_extend
            )

    # Score
    final_score = max(M[n, m], Ix[n, m], Iy[n, m])

    # Traceback
    align1, align2 = [], []
    i, j = n, m

    # Determine which matrix we're starting in
    if M[n, m] >= Ix[n, m] and M[n, m] >= Iy[n, m]:
        cur = 'M'
    elif Ix[n, m] >= Iy[n, m]:
        cur = 'Ix'
    else:
        cur = 'Iy'

    while i > 0 or j > 0:
        if cur == 'M':
            sc = match if seq1[i - 1] == seq2[j - 1] else mismatch
            align1.append(seq1[i - 1])
            align2.append(seq2[j - 1])
            prev_val = M[i, j] - sc
            if abs(M[i - 1, j - 1] - prev_val) < 1e-9:
                cur = 'M'
            elif abs(Ix[i - 1, j - 1] - prev_val) < 1e-9:
                cur = 'Ix'
            else:
                cur = 'Iy'
            i -= 1; j -= 1
        elif cur == 'Ix':
            align1.append(seq1[i - 1])
            align2.append('-')
            if abs(M[i - 1, j] + gap_open - Ix[i, j]) < 1e-9:
                cur = 'M'
            else:
                cur = 'Ix'
            i -= 1
        else:  # Iy
            align1.append('-')
            align2.append(seq2[j - 1])
            if abs(M[i, j - 1] + gap_open - Iy[i, j]) < 1e-9:
                cur = 'M'
            else:
                cur = 'Iy'
            j -= 1

    return int(final_score), ''.join(reversed(align1)), ''.join(reversed(align2))


# Tests
print("=== Linear vs. Affine gap penalty ===\n")
seq1 = 'ATCGATCG'
seq2 = 'ATCGTTTATCG'  # has a 3-base insertion

from curriculum.notebooks_draft import needleman_wunsch  # reference implementation


# Use the NW from NB04 for comparison
def needleman_wunsch(seq1, seq2, match=1, mismatch=-1, gap=-2):
    n, m = len(seq1), len(seq2)
    F = np.zeros((n+1, m+1), dtype=int)
    for i in range(n+1): F[i,0] = i*gap
    for j in range(m+1): F[0,j] = j*gap
    for i in range(1,n+1):
        for j in range(1,m+1):
            sc = match if seq1[i-1]==seq2[j-1] else mismatch
            F[i,j] = max(F[i-1,j-1]+sc, F[i-1,j]+gap, F[i,j-1]+gap)
    align1, align2 = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i>0 and j>0:
            sc = match if seq1[i-1]==seq2[j-1] else mismatch
            if F[i,j]==F[i-1,j-1]+sc:
                align1.append(seq1[i-1]); align2.append(seq2[j-1]); i-=1; j-=1; continue
        if i>0 and F[i,j]==F[i-1,j]+gap:
            align1.append(seq1[i-1]); align2.append('-'); i-=1
        else:
            align1.append('-'); align2.append(seq2[j-1]); j-=1
    return int(F[n,m]), ''.join(reversed(align1)), ''.join(reversed(align2))


score_lin, a1_lin, a2_lin = needleman_wunsch(seq1, seq2, match=1, mismatch=-1, gap=-2)
score_aff, a1_aff, a2_aff = affine_gap_nw(seq1, seq2, match=1, mismatch=-1, gap_open=-4, gap_extend=-1)

print(f"Sequences: '{seq1}' vs '{seq2}'\n")
print(f"Linear gap (gap=-2):    score={score_lin}")
print(f"  seq1: {a1_lin}")
print(f"  seq2: {a2_lin}")
print()
print(f"Affine gap (open=-4, extend=-1): score={score_aff}")
print(f"  seq1: {a1_aff}")
print(f"  seq2: {a2_aff}")

In [ ]:
# Step 7 — Visualization: gap penalty comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Cost of gap of length k for linear vs. affine
ax = axes[0]
k_vals = np.arange(1, 21)

linear_cost = -2 * k_vals
affine_cost = -4 + (-1) * (k_vals - 1)  # open + (k-1)*extend

ax.plot(k_vals, linear_cost, 'r-o', markersize=5, label='Linear (d=-2 per pos)')
ax.plot(k_vals, affine_cost, 'b-s', markersize=5, label='Affine (open=-4, extend=-1)')
ax.axhline(0, color='k', ls='--', alpha=0.3)
ax.set_xlabel('Gap length (k)')
ax.set_ylabel('Gap cost')
ax.set_title('Linear vs. Affine Gap Penalty\nAffine is cheaper for long gaps')
ax.legend()

# Annotate crossover point
# Linear: -2k; Affine: -4 + (k-1)(-1) = -3-k+1 = -k-3
# Equal at: -2k = -k-3 → k=3
ax.axvline(3, color='gray', ls=':', label='Crossover at k=3')
ax.text(3.2, -8, 'Crossover', fontsize=9)

# Panel B: Cost per position vs. gap length
ax2 = axes[1]
linear_per_pos = linear_cost / k_vals
affine_per_pos = affine_cost / k_vals

ax2.plot(k_vals, linear_per_pos, 'r-o', markersize=5, label='Linear (constant)')
ax2.plot(k_vals, affine_per_pos, 'b-s', markersize=5, label='Affine (decreasing)')
ax2.set_xlabel('Gap length (k)')
ax2.set_ylabel('Cost per position')
ax2.set_title('Cost per position\nAffine amortizes the open penalty')
ax2.legend()

plt.tight_layout()
plt.savefig('gap_penalties.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey insight: for k>3, affine gap is cheaper than linear.")
print("This discourages fragmenting a long indel into many short gaps.")

---
## Why three matrices?

The core question: when we're at cell $(i, j)$, and we're in a gap (say, gap in seq1),
we need to know whether this gap *just started* (pay $g_o$) or was *already in progress*
(pay only $g_e$). The only way to track this is to maintain a separate matrix that
records the best score *given that* we're currently in a gap of each type.

Matrix $I_x$: optimal score when seq1 has a gap at position $i$ (gap in seq1 → seq2[j] aligned to `-`)
Matrix $I_y$: optimal score when seq2 has a gap at position $j$
Matrix $M$: optimal score when ending with an aligned pair (match or mismatch)

---
## Step 8 — Exercises

See `exercises/07_affine_gap_exercises.md`.

1. By hand: compute the affine-gap alignment of 'ATCG' vs 'ATTTCG' with open=-3,
   extend=-1. Verify with your implementation.
2. What is the total cost of a gap of length 5 with open=-11, extend=-1?
   (BLAST protein defaults.)
3. Implement affine gap from scratch without reference. Test on 3 pairs.
4. At what gap length does affine (open=-4, extend=-1) become cheaper than
   linear (d=-2)? Derive algebraically.

---
## Step 10 — Quiz

1. What is the biological motivation for affine over linear gap penalties?
2. Name the three matrices in the affine-gap DP and what each represents.
3. What is the cost of a 4-gap insertion with open=-11, extend=-1?
4. Why does the affine-gap recurrence for $I_x$ only reference $M[i-1][j]$ and
   $I_x[i-1][j]$, not $I_y[i-1][j]$?
5. In BLAST, what are the default gap open and extend penalties for blastp?

---
## Step 12 — Reflection

> *[Explain in one paragraph: why do we need three matrices for affine gap, but only
> one for linear gap? What information is lost in the one-matrix formulation?]*

---
*Next: `08_how_blast_actually_works.ipynb`*